In [0]:
import pyspark.sql.functions as F
from pyspark.sql import Window

df_crm_customer_info = spark.read.table("sales_sports_pyspark.01_silver_pyspark.crm_customer_info")
df_crm_product_info = spark.read.table("sales_sports_pyspark.01_silver_pyspark.crm_product_info")
df_crm_sales_details = spark.read.table("sales_sports_pyspark.01_silver_pyspark.crm_sales_details")
df_erp_cust_az12 = spark.read.table("sales_sports_pyspark.01_silver_pyspark.erp_cust_az12")
df_erp_loc_a101 = spark.read.table("sales_sports_pyspark.01_silver_pyspark.erp_loc_a101")
df_erp_px_cat_g1v2 = spark.read.table("sales_sports_pyspark.01_silver_pyspark.erp_px_cat_g1v2")

df_dim_customer_info = df_crm_customer_info.join(df_erp_cust_az12, df_crm_customer_info["cst_key"] == df_erp_cust_az12["cid"], how="left")
df_dim_customer_info = df_dim_customer_info.join(df_erp_loc_a101, on = "cid", how="left")

df_dim_customer_info = df_dim_customer_info.withColumn(
    "cst_gndr",
    F.when((F.col("cst_gndr") != 'N/A') | (F.col("cst_gndr") != 'n/a'), (F.col("cst_gndr")))
     .otherwise(F.coalesce(F.col("gen"), F.lit("N/A")))
)

df_dim_customer_info = df_dim_customer_info.drop("cid", "gen")
df_dim_customer_info = df_dim_customer_info.withColumnRenamed("bdate", "birthdate")
df_dim_customer_info = df_dim_customer_info.withColumnRenamed("cntry", "country")
df_dim_customer_info = df_dim_customer_info.withColumnRenamed("cst_create_date", "creation_date")
df_dim_customer_info = df_dim_customer_info.withColumnRenamed("cst_id", "customer_id")
df_dim_customer_info = df_dim_customer_info.withColumnRenamed("cst_key", "customer_number")
df_dim_customer_info = df_dim_customer_info.withColumnRenamed("cst_lastname", "last_name")
df_dim_customer_info = df_dim_customer_info.withColumnRenamed("cst_marital_status", "marital_status")
df_dim_customer_info = df_dim_customer_info.withColumnRenamed("cst_firstname", "first_name")
df_dim_customer_info = df_dim_customer_info.withColumnRenamed("cst_gndr", "gender")


df_dim_customer_info.write.format("delta").mode("overwrite").saveAsTable("sales_sports_pyspark.02_gold_pyspark.dim_customer_info")


In [0]:
df_crm_product_info = df_crm_product_info.filter(F.col("prd_end_dt").isNull())
df_crm_product_info = df_crm_product_info.drop("prd_end_dt")
df_dim_product = df_crm_product_info.join(df_erp_px_cat_g1v2, df_crm_product_info["cat_id"] == df_erp_px_cat_g1v2["id"], how="left")
df_dim_product = df_dim_product.drop("ID")
df_dim_product = df_dim_product.withColumnRenamed("prd_id", "product_id")
df_dim_product = df_dim_product.withColumnRenamed("cat_id", "category_id")
df_dim_product = df_dim_product.withColumnRenamed("prd_key", "product_number")
df_dim_product = df_dim_product.withColumnRenamed("prd_nm", "product_name")
df_dim_product = df_dim_product.withColumnRenamed("prd_cost", "product_cost")
df_dim_product = df_dim_product.withColumnRenamed("prd_line", "product_line")
df_dim_product = df_dim_product.withColumnRenamed("prd_start_dt", "product_start_date")
df_dim_product = df_dim_product.withColumnRenamed("CAT", "category")
df_dim_product = df_dim_product.withColumnRenamed("SUBCAT", "subcategory")
df_dim_product = df_dim_product.withColumnRenamed("MAINTENANCE", "maintenance")

df_dim_product.write.format("delta").mode("overwrite").saveAsTable("sales_sports_pyspark.02_gold_pyspark.dim_product_info")

In [0]:
df_fact_sales = df_crm_sales_details.join(df_dim_product, df_crm_sales_details["sls_prd_key"] == df_dim_product["product_number"], how="left")
df_fact_sales = df_fact_sales.join(df_dim_customer_info, df_fact_sales["sls_cust_id"] == df_dim_customer_info["customer_id"], how="left")
df_fact_sales = df_fact_sales.drop("sls_prd_key", "sls_cust_id", "country", "birthdate", "category", "subcategory", "marital_status", "maintenance", "gender", "first_name", "last_name", "category_id", "creation_date", "product_line", "product_name", "product_start_date", "customer_number", "product_number", "product_cost")
df_fact_sales = df_fact_sales.withColumnRenamed("sls_order_dt", "order_date")
df_fact_sales = df_fact_sales.withColumnRenamed("sls_ord_num", "order_number")
df_fact_sales = df_fact_sales.withColumnRenamed("sls_ship_dt", "shipping_date")
df_fact_sales = df_fact_sales.withColumnRenamed("sls_due_dt", "due_date")
df_fact_sales = df_fact_sales.withColumnRenamed("sls_sales", "sales_amount")
df_fact_sales = df_fact_sales.withColumnRenamed("sls_quantity", "sales_quantity")
df_fact_sales = df_fact_sales.withColumnRenamed("sls_price", "price")

df_fact_sales.write.format("delta").mode("overwrite").saveAsTable("sales_sports_pyspark.02_gold_pyspark.fact_sales")